# Pandas: practical tabular data analysis

Pandas is the main tool for working with tables in this course. This notebook shows you a common workflow: load data, inspect it, clean it, transform it, summarize it, and prepare it for analysis.

The dataset contains **69 entirely fictional AAI students**. It was generated for artificially and it is not real University of Bayreuth data.

## 1. Imports and portable data loading

The conventional alias for Pandas is `pd`. `pathlib.Path` helps us find the CSV whether Jupyter starts in the repository root or in `00_Setup`.

In [ ]:
from pathlib import Path
import pandas as pd


candidate_paths = [
  parent / "data" / "aai_student_data.csv"
  for parent in (Path.cwd(), *Path.cwd().parents)
]


data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
  raise FileNotFoundError("Could not find data/aai_student_data.csv in the repository.")

print(f"Loading: {data_path.resolve()}")

## 2. Series and DataFrames

A `Series` is one labeled column. A `DataFrame` is a table of aligned Series. You can construct both directly from Python objects.

In [ ]:
scores = pd.Series([99, 69, 3], index=["Julika", "Leopold", "Victor"], name="exam_score")
print(scores)
print(scores.mean())

tiny_frame = pd.DataFrame({
  "student": ["Ada", "Grace", "Margaret"],
  "score": [67, 69, 99],
  "passed": [True, True, True],
})
display(tiny_frame)

In [ ]:
# In jupyter notebooks, you can also just type the variable name to display it:
tiny_frame

### Load a CSV

`read_csv()` is one of the functions you will use most often. The returned object is a DataFrame.

In [ ]:
students = pd.read_csv(data_path)
students.head()

## 3. First inspection

Never begin an analysis by trusting the file name. Check rows, columns, types, missing values, and plausible ranges first.

In [ ]:
print("Shape:", students.shape)
print("Columns:", students.columns.tolist())
print("Index:", students.index)
students.info()

In [ ]:
students.describe(include="all")

In [ ]:
students.describe(include="all").T

In [ ]:
display(students.head(3))
print(students.tail(3))
print(students.sample(3, random_state=420))

### Your turn: inspect

Display seven randomly selected rows reproducibly and calculate the number of missing values per column.

In [ ]:
### --- Start Your Turn --- ###
inspection_sample  = None # TODO: use sample() with random_state=420
missing_per_column = None # TODO: combine isna() and sum()
### --- Start Your Turn --- ###

print(inspection_sample)
print(missing_per_column)

## 4. Selecting rows and columns

Use brackets for one or more columns, `.loc` for label-based selection, and `.iloc` for position-based selection.

In [ ]:
score_series = students["exam_score"]
small_table = students[["student_id", "study_program", "exam_score"]]

print(score_series.head())
print(small_table.head())

In [ ]:
print(students.loc[0:2, ["student_id", "exam_score"]]) # Labels 0, 1, and 2 are included. Human-logical indexing.
print(students.iloc[0:3, [0, 7]]) # Positions 0, 1, and 2. Stop is excluded. Python-logical indexing.

Boolean masks filter rows. Parenthesize each comparison when combining masks with `&` or `|`.

In [ ]:
hard_working = students[
  (students["weekly_study_hours"] >= 8)
  & (students["assignments_completed"] >= 8)
]

business_or_math = students[
  students["study_program"].isin(["Business Administration", "Mathematics"])
]

print(hard_working[["student_id", "weekly_study_hours", "assignments_completed"]].head())
print(business_or_math["study_program"].value_counts())

### Your turn: filter

Select the ID, program, and score of students in tutorial group `T2` who scored at least `69`.

In [ ]:
### --- Start Your Turn --- ###
selected_students = None # TODO: filter with .loc and select three columns
### --- End Your Turn --- ###

print(selected_students)

## 5. Sorting, renaming, and derived columns

In [ ]:
top_students = students.sort_values(
  by=["exam_score", "weekly_study_hours"],
  ascending=[False, True],
)
top_students[["student_id", "exam_score", "weekly_study_hours"]].head()

In [ ]:
analysis = students.rename(columns={"exam_score": "score"}).copy()
analysis["study_minutes"] = analysis["weekly_study_hours"] * 60
analysis["score_per_assignment"] = analysis["score"] / analysis["assignments_completed"].replace(0, pd.NA)
analysis[["student_id", "score", "study_minutes", "score_per_assignment"]].head()

Use `.assign()` for a readable transformation chain. Each step returns a new DataFrame.

In [ ]:
enriched = students.assign(
  attendance_fraction=lambda frame: frame["attendance_percent"] / 100,
  passed_label=lambda frame: frame["passed"].map({True: "passed", False: "not passed"}),
)
enriched[["student_id", "attendance_fraction", "passed_label"]].head()

## 6. Missing values, duplicates, and types

The fictional data intentionally contains a few missing values. Decide whether to keep, remove, or impute them based on the analysis---not because `.dropna()` was the first method you remembered.

In [ ]:
students.isna().sum()

In [ ]:
clean_students = students.copy()
median_hours = clean_students["weekly_study_hours"].median()
clean_students["weekly_study_hours"] = clean_students["weekly_study_hours"].fillna(median_hours)
clean_students["python_experience"] = clean_students["python_experience"].fillna("Unknown")

print(clean_students.isna().sum())

Create duplicates in memory so the source CSV stays clean. `duplicated()` identifies them. `drop_duplicates()` removes them.

In [ ]:
with_duplicate = pd.concat([students, students.iloc[[0]]], ignore_index=True)
print("Duplicate rows:", with_duplicate.duplicated().sum())

without_duplicates = with_duplicate.drop_duplicates()
print(without_duplicates.shape)

Convert types explicitly. `errors='coerce'` turns invalid numeric text into missing values. Parse date text with `to_datetime()`. Categories can save memory and communicate a finite set of labels.

In [ ]:
typed_students = clean_students.copy()
typed_students["enrollment_date"] = pd.to_datetime(typed_students["enrollment_date"])
typed_students["tutorial_group"] = typed_students["tutorial_group"].astype("category")
typed_students["exam_score"] = pd.to_numeric(typed_students["exam_score"], errors="coerce")

print(typed_students.dtypes)
print(typed_students["enrollment_date"].dt.day_name().head())

### Safe assignment

Avoid chained assignment such as `students[mask]['passed'] = True`. It may modify a temporary object. Use `.loc[rows, columns]` on an explicit copy.

In [ ]:
adjusted_students = students.copy()
high_attendance = adjusted_students["attendance_percent"] >= 95
adjusted_students.loc[high_attendance, "attendance_label"] = "excellent"
adjusted_students.loc[~high_attendance, "attendance_label"] = "regular"
adjusted_students[["attendance_percent", "attendance_label"]].head()

### Your turn: clean

Create a copy named `exercise_clean` and fill missing Python experience with `"Not reported"`.

In [ ]:
exercise_clean = students.copy()

### --- Start Your Turn --- ###
# TODO: fill missing values in python_experience on exercise_clean

### --- End Your Turn --- ###

print(exercise_clean["python_experience"].value_counts(dropna=False))

## 7. Strings, mapping, and `apply`

The `.str` accessor applies vectorized string operations. `map()` is ideal for value-to-value mappings. Prefer vectorized expressions. Use `apply()` when row logic is genuinely clearer.

In [ ]:
string_demo = students[["student_id", "study_program"]].copy()
string_demo["program_lower"] = string_demo["study_program"].str.lower()
string_demo["mentions_science"] = string_demo["study_program"].str.contains("Science", case=False)
string_demo.head()

In [ ]:
experience_order = {
  "None": 0,
  "Basic": 1,
  "Intermediate": 2,
  "Advanced": 3,
}
mapped_students = students.copy()
mapped_students["experience_level"] = mapped_students["python_experience"].map(experience_order)
mapped_students[["python_experience", "experience_level"]].head(10)

In [ ]:
def performance_label(row: pd.Series) -> str:
  if row["exam_score"] >= 80 and row["assignments_completed"] >= 8:
    return "strong and consistent"
  if row["exam_score"] >= 50:
    return "passed"
  return "needs another round"


labelled_students = students.copy()
labelled_students["performance_label"] = labelled_students.apply(performance_label, axis=1)
labelled_students["performance_label"].value_counts()

## 8. Grouping and aggregation

The split-apply-combine pattern groups rows, calculates summaries, and combines the results.

In [ ]:
program_summary = (
  students
  .groupby("study_program", as_index=False)
  .agg(
    students=("student_id", "count"),
    average_score=("exam_score", "mean"),
    median_study_hours=("weekly_study_hours", "median"),
    pass_rate=("passed", "mean"),
  )
  .sort_values("average_score", ascending=False)
)

program_summary.round(2)

In [ ]:
students["python_experience"].value_counts(dropna=False, normalize=True).mul(100).round(1)

### Your turn: aggregate

For each tutorial group, calculate the number of students and the median exam score.

In [ ]:
### --- Start Your Turn --- ###
tutorial_summary = None # TODO: groupby() and agg()
#### --- End Your Turn --- ###

print(tutorial_summary)

## 9. Combining and reshaping tables

`merge()` joins tables using key columns. Validate the expected relationship when possible.

In [ ]:
program_support = pd.DataFrame({
  "study_program": [
    "Computer Science",
    "Mathematics",
    "Business Administration",
    "Sports Economics",
    "Philosophy",
  ],
  "support_contact": ["CS Lab", "Math Lab", "Business Lab", "Sports Lab", "Humanities Lab"],
})

students_with_support = students.merge(
  program_support,
  on="study_program",
  how="left",
  validate="many_to_one",
)
students_with_support[["student_id", "study_program", "support_contact"]].head()

`concat()` stacks compatible objects. Here we split and recombine rows only to demonstrate the operation.

In [ ]:
first_half = students.iloc[:35]
second_half = students.iloc[35:]
recombined = pd.concat([first_half, second_half], ignore_index=True)
print(recombined.shape)
print(recombined["student_id"].is_unique)

Pivot tables summarize combinations of categories. `melt()` converts wide columns into a longer, tidy representation.

In [ ]:
score_pivot = pd.pivot_table(
  students,
  index="study_program",
  columns="tutorial_group",
  values="exam_score",
  aggfunc="mean",
).round(1)
score_pivot

In [ ]:
long_metrics = students.melt(
  id_vars=["student_id"],
  value_vars=["weekly_study_hours", "attendance_percent", "exam_score"],
  var_name="metric",
  value_name="value",
)
long_metrics.head()

### Your turn: reshape

Create a pivot table of mean `weekly_study_hours` by study program and tutorial group.

In [ ]:
### --- Start Your Turn --- ###
study_hours_pivot = None # TODO: use pd.pivot_table()
### --- End Your Turn --- ###

print(study_hours_pivot)

## 10. Compact exploratory workflow

A first pass should answer: Is the shape plausible? Which data is missing? What are the distributions and groups? Which relationships deserve a plot or a model? Descriptive patterns are not automatically causal conclusions.

In [ ]:
eda_summary = {
  "rows": len(students),
  "columns": students.shape[1],
  "missing_values": int(students.isna().sum().sum()),
  "mean_score": round(students["exam_score"].mean(), 2),
  "pass_rate_percent": round(students["passed"].mean() * 100, 1),
  "most_common_program": students["study_program"].mode().iat[0],
}
eda_summary

In [ ]:
numeric_relationships = students[
  ["weekly_study_hours", "attendance_percent", "assignments_completed", "exam_score"]
].corr(numeric_only=True)
numeric_relationships.round(2)

### Exporting results

`to_csv()` writes a DataFrame.

```python
program_summary.to_csv("program_summary.csv", index=False)
```

Also explore `read_excel()`, `to_excel()`, `read_parquet()`, and database connectors when your project needs them.

## Takeaway

A reliable Pandas workflow is:

1. Load and inspect.
2. Select and validate.
3. Clean explicitly.
4. Transform without chained assignment.
5. Group, combine, and reshape.
6. Communicate assumptions and limitations.

References: [Pandas getting started](https://pandas.pydata.org/docs/getting_started/index.html) and [Pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html).

Next: **00_04_Numpy.ipynb**.